# Synthetic RAG Data Generation

This notebook walks through the steps to automatically generate question-answer pairs for training a search / RAG model from a data corpus.

Data corpus can be any collection of text documents, such as:

- markdown files (notion / developer docs / obsidian vault)
- pdfs
- emails
- text / slack messages
- mix of various sources

This notebook currently only supports parsing markdown files. Support for other modalities will be added with time.

The main process for generating synthetic data for search / RAG:

1. Sample a section of a document as a reference chunk (ideally it contains interesting information)
2. Tell an LLM to generate a reasonable search query + answer pair that could only be found in this sample.

With this process, we will get a list of <question, answer, reference chunk(s)> which will serve as the train / eval dataset.
Reference chunk can be used in the reward function to determine if the search agent has retrieved the proper context.

This notebook covers the basics to get a decent v1 dataset - feel free to extend this notebook to generate / filter for higher quality questions :)


## 0. Setup

### Google Colab Setup

This section sets up the environment for running in Google Colab:

1. Clone the synthetic-data-prep library
2. Install dependencies
3. Configure API credentials


In [1]:
# Clone the repository
# !git clone https://github.com/cgft-io/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib

# # Install the library with dev dependencies
# %pip install -e .[dev]

In [9]:
# This is only used internally for development

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Add the src directory to the path for local imports
repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Document Chunking

RAG systems retrieve chunks, not entire documents. So our first step is splitting documents into pieces that can be indexed and retrieved.

We went with chunk size of min 1024 chars with a max size of 2048 as a reasonable starting point, though you should experiment based on your use case.


In [10]:
from synthetic_data_prep.chunkers.inspector import ChunkInspector
from synthetic_data_prep.chunkers.markdown import MarkdownChunker

# ===== MODIFY HERE: Replace the placeholder / tweak the configs =====
DOCS_PATH = "../samples/posthog"
# DOCS_PATH = "./samples/cgft-notion"
# DOCS_PATH = "./samples/based.cooking"
# DOCS_PATH = "./samples/learning-notes"

# Our markdown chunker break down a document into chunks by the header
# If a chunk is too small (< MIN_CHARS), we will try to fuse the chunk
# with its neighbor while not exceeding MAX_CHARS
# If a chunk is too big (> MAX_CHARS), we will break it down
# into smaller chunks while using OVERLAP_CHARS to retain context of the prev chunk.

MIN_CHARS = 1024
MAX_CHARS = 2048
OVERLAP_CHARS = 128
# ===== END OF MODIFY SECTION =====

# Configure chunker
chunker = MarkdownChunker(min_char=MIN_CHARS, max_char=MAX_CHARS, chunk_overlap=OVERLAP_CHARS)

# Chunk folder - returns ChunkCollection
collection = chunker.chunk_folder(DOCS_PATH, file_extensions=[".md", ".mdx"])

# Use inspector to analyze chunks
inspector = ChunkInspector(collection)
inspector.summary(max_depth=3, max_files_per_folder=4)

ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3 chunks)
│   ├── index.mdx (2 chunks)
│   ├── 

In [11]:
# UNCOMMENT the code below to inspect the quality of the chunking

# # Inspect a random chunk
# print(inspector.sample_chunk(show_context_before=True, show_context_after=True))

# # Inspect a random file
# print(inspector.sample_file(max_chunks=10))

# # Inspect a specific file
# print(inspector.read_file("product-toolkit.mdx", max_chunks=10))


# UNCOMMENT the code below to save chunks to a file
# save_chunks(collection, "outputs/chunks.yaml")

# UNCOMMENT the code below to load chunks from a file
# collection = load_chunks("outputs/chunks.yaml")
# inspector = ChunkInspector(collection)
# inspector.summary(max_depth=3, max_files_per_folder=4)

Feel free to tweak the configuration above to achieve better chunking


## 2. Upload Corpus

Upload your chunks to our corpus API for BM25 search and evaluation. This enables:

- Fast BM25 search over your chunked documents
- Integration with the training pipeline
- Evaluation of retrieval quality

**Note:** Chunks cannot be deleted individually. If you need to redo the upload, delete the corpus and create a new one.


In [63]:
from synthetic_data_prep.corpus.corpora.source import CorporaChunkSource

# ===== MODIFY HERE: Configure your API and corpus settings =====
API_KEY = "sk_RZEmogqiKirhhDCtipwhSKeqFqnuLUjoxfTQqTkRvNWyuMFOoJkEJxWgJnwoobKS"
CORPUS_NAME = "posthog"
CORPUS_BASE_URL = "https://app.cgft.io"
# ===== END OF MODIFY SECTION =====

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=CORPUS_BASE_URL)

In [64]:
source.populate_from_folder(DOCS_PATH)
collection = source.collection  # available for inspection in cells below

Chunking documents from ../samples/posthog...
ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3

Uploading chunks:   0%|          | 0/33 [00:00<?, ?batch/s]


Upload complete! Inserted: 3225


In [17]:
# Test the BM25 search functionality

# ===== MODIFY HERE: Test your search queries =====
TEST_QUERIES = [
    "how to feature flag",
    "setup reverse proxy",
    "nextjs installation",
]
# ===== END OF MODIFY SECTION =====

print("Testing search queries:\n")
for query in TEST_QUERIES:
    results = source._client.search(corpus_id=source.corpus_id, query=query, limit=3)

    print(f"Query: '{query}'")
    print(f"Found {results.total} results")

    for i, chunk_a in enumerate(results.results[:3], 1):
        preview = chunk_a.content[:100].replace("\n", " ")
        print(f"  {i}. (score: {chunk_a.score:.3f}) {preview}...")
    print()

Testing search queries:

Query: 'how to feature flag'
Found 2661 results
  1. (score: 9.868) ## Framework guides   - [How to set up Android feature flags](/tutorials/android-feature-flags) - [H...
  2. (score: 9.559) - [How to set up Go feature flags](/tutorials/go-feature-flags) - [How to set up Python feature flag...
  3. (score: 9.557) --- title: Tutorials and guides sidebar: Docs showTitle: true --- Got a question which isn't answere...

Query: 'setup reverse proxy'
Found 341 results
  1. (score: 15.885) --- title: Deploy a reverse proxy sidebar: Docs showTitle: true ---   import ProxyPlatforms from './...
  2. (score: 14.204) ## Option 1: Basic setup   This setup proxies all PostHog requests through a dedicated subdomain.   ...
  3. (score: 13.883) --- title: Caddy reverse proxy sidebar: Docs showTitle: true showStepsToc: true platformLogo: caddy ...

Query: 'nextjs installation'
Found 552 results
  1. (score: 10.979) --- title: Next.js experiments installation platformLogo: nextj

## 3. Q&A Generation

We want to generate diverse and realistic questions that test different retrieval and reasoning capabilities. We group the questions into 2 main types:

**Single-Hop Questions** — One chunk contains the complete answer

- `postgres jsonb index syntax` -> SQL reference doc
- `company pto policy parental leave` -> HR handbook section
- `sourdough starter feeding ratio` -> recipe instructions

**Multi-Hop Questions** — Information from multiple chunks required

- `error handling in tanstack query mutations` -> TanStack Query overview -> mutations guide -> error boundary section
- `who approved the Q2 marketing budget` -> budget doc -> approval workflow -> executive sign-off

We'll tackle each question type in order, validating as we go.


### 3.1 LLM Client Setup


In [57]:
from openai import OpenAI

# ===== MODIFY HERE: Configure your LLM settings =====
# Note: We provide access to a few standard models + some budget for you but feel free to BYOM
LLM_API_KEY = API_KEY
MODEL_ENDPOINT = "http://app.cgft.io/api/llm"
MODEL = "grok-4-fast-non-reasoning"  # this model is the fastest with decent quality for our labelling task
# ===== END OF MODIFY SECTION =====

client = OpenAI(base_url=MODEL_ENDPOINT, api_key=LLM_API_KEY)

### 3.2 Generate Corpus Context

To create a corpus-level context, we can get the LLM to summarize the corpus using some top level / indexing chunks + sample a few random chunks.

It is also helpful for us (owner of the corpus) to provide some short context and example queries.


In [10]:
import random

from synthetic_data_prep.qa_generation.helpers import print_sections, render_template
from synthetic_data_prep.qa_generation.response_parsers import parse_corpus_summary_response

# ===== MODIFY HERE: Give your short description of the corpus and some example questions =====
YOUR_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

NUM_TOP_LEVEL_CHUNK_SAMPLES = 4
NUM_RANDOM_SAMPLES = 4
# ===== END OF MODIFY SECTION =====

# CORPUS Analysis System and User prompts
corpus_system_prompt = """You are a technical analyst specializing in document corpus analysis.
Your goal is to understand the overall themes, content type, and typical query that a user might search for.

Based on the context and thoughts, form an understanding of the corpus, and then provide a short summary and example search queries. Weigh the user's input if provided.

Guideline:
- Try to generalize and identify the overall theme of the corpus.
- Use your context of the theme, domain, etc. to guess the persona and purpose of searching this corpus
- i.e.
    - student finding their learning notes
    - developer looking up API documentation
    - journalist researching a topic
    - business analyst gathering market research
- Use that persona to guess the types of queries they might perform.

For the summary:
- First line - corpus themes (documentation, tutorials, reference, etc.)
- Second line - content domain (technical, business, scientific, etc.)
- Third line - user persona and purpose (likely developer looking up API documentation)
- Do NOT cite specific chunk content in the summary.

For the example queries:
- Provide 5-10 realistic example queries a user might search for in this corpus.
- Use the inferred user persona and purpose to guide the query style.
- Queries can have incomplete information, as often users do not remember full context.

Return JSON with:
- thoughts: Your analysis and reasoning here
- summary: our summary here (3 lines as described above)
- example_queries: List of example queries in the form of ["query1", "query2", ...]

"""

corpus_user_template = """Analyze the following document corpus:

<user_context>
{user_context}
</user_context>

<top_level_chunks>
{top_level_content}
</top_level_chunks>

<random_sampled_chunks>
{random_content}
</random_sampled_chunks>

Return your analysis as JSON with keys: thoughts, summary, example_queries
"""


async def generate_summary_and_queries(
    source,
    your_description: str,
    example_queries: list[str],
    num_top_level: int,
    num_random: int,
) -> tuple[str, str, str]:
    top_level = source.get_top_level_chunks()
    sampled_top_level = random.sample(top_level, min(num_top_level, len(top_level)))
    sampled_random = source.sample_chunks(num_random, min_chars=400)

    variables = {
        "user_context": f"Description: {your_description}\nExample queries provided by user: {', '.join(example_queries)}",
        "top_level_content": "\n\n".join([chunk.chunk_str() for chunk in sampled_top_level]),
        "random_content": "\n\n".join([chunk.chunk_str() for chunk in sampled_random]),
    }

    corpus_user_prompt = render_template(corpus_user_template, variables)

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": corpus_system_prompt},
            {"role": "user", "content": corpus_user_prompt},
        ],
    )

    return corpus_system_prompt, corpus_user_prompt, completion.choices[0].message.content or ""


print("Generating corpus summary and example queries. This should take ~10+ seconds...\n")

system_prompt, user_prompt, response = await generate_summary_and_queries(
    source,
    
    YOUR_DESCRIPTION,
    EXAMPLE_QUERIES,
    NUM_TOP_LEVEL_CHUNK_SAMPLES,
    NUM_RANDOM_SAMPLES,
)

print_sections(
    # # Uncomment the following to inspect the system / user prompt
    # ( "System Prompt", system_prompt ),
    # ("User Prompt", user_prompt),
    ("Response", response),
)

corpus_summary, example_queries = parse_corpus_summary_response(response)
corpus_queries = "\n".join([f"- {q}" for q in example_queries])
corpus_context = {"corpus_summary": corpus_summary, "corpus_queries": corpus_queries}

Generating corpus summary and example queries. This should take ~10+ seconds...

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                       Response                                                       │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
{
  "thoughts": "The corpus consists of PostHog documentation, including a glossary of terms related to analytics, product management, and compliance; guides on feature flags and use cases like kill switches; detailed instructions for product analytics tools such as funnels, trends, and filters; setup for integrations like batch exports to Databricks; and JavaScript library usage for event handling and redaction. The user context confirms it's PostHog docs and company policy, with example queries focused on feature flags, proxy setup, and installatio

### 3.3 Single-Hop Q&A Generation

Single-hop questions are the simplest to work with. All you need is a single reference chunk that contains meaningful information, and you can ask the model to generate query <> answer pairs.

On top of the reference chunk, providing corpus-level summary + example questions also helps with creating realistic questions.


In [11]:
from synthetic_data_prep.chunkers.models import Chunk
from synthetic_data_prep.qa_generation.response_parsers import parse_single_hop_response

# ===== MODIFY HERE: feel free to adjust the num preview chars for the prev / after context =====
CONTEXT_PREVIEW_CHARS = 200
# ===== END OF MODIFY SECTION =====

single_hop_system_template = """You are generating realistic search queries for a RAG system.

Corpus summary:
{corpus_summary}

Example queries:
{corpus_queries}

Guide:
- When generating queries, keep them terse, keyword-heavy like how real users would search. i.e.:
    - "k8s pod memory limits configuration"
    - "python async await best practices"
    - "quarterly revenue breakdown Q3"
- Queries / answers does not need to encompass the whole chunk. Query just need to target specific piece in the chunk that a user would likely want to know
- Query does not have to completely target all keywords in the chunk since users often only have partial recollection of the information, which is why they are searching
- The query should be answerable from the provided chunk alone
- Paraphrase the keywords / use synonyms in the query to make it natural and varied
- Rank and place your best question first if multiple q&a pairs are generated

Return JSON with:
- keywords: Relevant keywords that a user might search for in the chunk
- confidence: "low" | "mid" | "high";  use "low" if chunk has no meaningful information (too generic, boilerplate, or navigation-only)
- qa_pairs: List of query and answer pairs in the form of [{{"query": "q1", "answer": "a1"}}, ...]
"""

single_hop_system_prompt = render_template(single_hop_system_template, corpus_context)

single_hop_user_template = """Generate a single-hop search q&a pairs based on the following chunk:

<context_before>
{prev_chunk_preview}
</context_before>

<main_chunk>
{chunk_content}
</main_chunk>

<context_after>
{next_chunk_preview}
</context_after>

Context before and after are only provided as additional context. Q&A should only target main chunk content.

Return JSON with keys: keywords, confidence, qa_pairs
"""


def generate_single_hop_qa(
    chunk: Chunk,
    client,
    model: str,
    context_preview_chars: int,
) -> tuple[str, str, str]:
    # Build the user prompt with chunk and context
    ctx = source.get_chunk_with_context(chunk, max_chars=context_preview_chars)
    single_hop_user_prompt = render_template(single_hop_user_template, ctx)

    # Call the LLM
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": single_hop_system_prompt},
            {"role": "user", "content": single_hop_user_prompt},
        ],
    )

    response = completion.choices[0].message.content or ""

    return single_hop_system_prompt, single_hop_user_prompt, response


# Sample 2 random chunks to inspect synthetic q&a quality
sample_chunks = source.sample_chunks(2, min_chars=400)

for i, chunk_a in enumerate(sample_chunks, 1):
    print(f"{'=' * 120}\nGenerating single-hop Q&A response should take ~10+ seconds...\n")

    system_prompt, user_prompt, response = generate_single_hop_qa(
        chunk_a, client, MODEL, CONTEXT_PREVIEW_CHARS
    )

    confidence, qa_pairs = parse_single_hop_response(response)

    print_sections(
        # (f"SAMPLE {i} of {len(sample_chunks)} - System Prompt", system_prompt),
        (f"SAMPLE {i} of {len(sample_chunks)} - User Prompt", user_prompt),
        (f"SAMPLE {i} of {len(sample_chunks)} - Response", response),
        # (
        #     f"SAMPLE {i} of {len(sample_chunks)} - Parsed Results",
        #     f"Confidence: {confidence}\nQ&A Pairs: {qa_pairs}",
        # ),
    )

Generating single-hop Q&A response should take ~10+ seconds...

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                             SAMPLE 1 of 2 - User Prompt                                              │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
Generate a single-hop search q&a pairs based on the following chunk:

<context_before>
(No previous chunk)
</context_before>

<main_chunk>
{
  "file": "cdp/sources/_snippets/source-shopify.mdx",
  "h2": "In Shopify",
  "index": 0
}
The Shopify connector can sync your Shopify store data into PostHog.  
To sync Shopify data:

## In Shopify  
1. Go to your Shopify store's admin panel at `https://admin.shopify.com/store/your-store-id`
2. Click **Settings**.
3. Click **Apps and sales channels**.
4. Click **Develop apps**.
5. **IMPORTANT:** Click **Build apps in Dev Dashbo

### 3.4 Multi-Hop Q&A Generation

Multi-hop questions require information from multiple chunks that share common topics/entities. Here is the process:

1. **Generate search queries** - For a given chunk, generate queries to find related chunks
2. **Search corpus** - Use BM25 to find candidate related chunks
3. **Filter and rank** - Remove adjacent chunks, rank by query overlap and score
4. **Validate connections** - Check if chunks have meaningful dependencies and generate Q&A pairs


In [12]:
from synthetic_data_prep.qa_generation.response_parsers import (
    parse_multi_hop_validation_response,
    parse_related_queries_response,
)

# ===== CONFIGURATION =====
CONTEXT_PREVIEW_CHARS_RELATED = 200

TOP_K_BM25_RESULTS = 5  # Number of top chunks to retrieve for related chunk search
TOP_RELATED_CHUNKS = 3  # Number of related chunks to perform multi-hop qa generation

# =========================


# Prompt for finding related chunks
related_chunk_system_template = """You are generating BM25 search queries to find chunks that have meaningful relationships with the given chunk.

Corpus summary:
{corpus_summary}

## BM25 Behavior
BM25 matches exact keywords, weighted by rarity. This means:
- Specific/rare terms (product names, technical terms, unique phrases) are powerful
- Common corpus terms (e.g., "API", "data", "system") barely help
- BM25 won't match synonyms: "k8s" won't find "Kubernetes"
- Shorter, focused queries often outperform long ones

## Query Strategies

**1. Entity-focused**: Target specific named things that might appear elsewhere
  - Product/tool names: "Redis", "Workday", "Stripe"
  - Internal terms: "Project Atlas", "Q3 planning", "customer churn analysis"
  - Document names: "migration guide", "onboarding checklist"

**2. Reference-chasing**: If this chunk mentions other docs/sections, query for them
  - "see the deployment guide" → query: "deployment guide"
  - "as discussed in Q2 review" → query: "Q2 review"

**3. Inverse references**: Query for terms that other chunks might use to reference this one
  - If this is the Redis setup guide → query: "Redis setup", "Redis configuration"
  - If this covers auth flow → query: "authentication", "OAuth implementation"

**5. Synonym/variant expansion**: Generate alternate phrasings for key concepts
  - "Kubernetes" + "k8s"
  - "authentication" + "auth" + "login"
  - "configuration" + "config" + "setup"

## Query Format
- Prefer specific terms over generic ones
- Include both the canonical term and common variants
- Each query should target a *different* potential related chunk
- If the chunk is boilerplate (e.g., empty template, generic footer), set confidence to "low" and generate few/no queries

Return JSON with:
- keywords: Distinctive terms from this chunk likely to appear in related chunks
- confidence: "low" | "mid" | "high" - based on how much unique, linkable content exists
- queries: ["q1", "q2", ...] - diverse queries targeting different potential relationships
"""

related_chunk_system_prompt = render_template(related_chunk_system_template, corpus_context)

related_chunk_user_template = """Generate search queries based on this chunk to find other relevant chunks:

<context_before>
{prev_chunk_preview}
</context_before>

<main_chunk>
{chunk_content}
</main_chunk>

<context_after>
{next_chunk_preview}
</context_after>

The before/after context is provided only as additional context. Queries should target content from the main chunk only.

Return JSON with keys: keywords, confidence, queries
"""


# Prompt for validating chunk connections and generating QA
multi_hop_system_template = """You are analyzing whether two chunks have a meaningful dependency.

Corpus summary:
{corpus_summary}

Example queries from corpus:
{corpus_queries}

## Task

Analyze two chunks to determine if a meaningful relationship exists, then generate multi-hop questions that exploit that relationship.

**Terminology:**
- **Source chunk**: Contains the reference, pointer, or entry point
- **Target chunk**: Contains the referenced content, details, or destination

## Overall Notes:
- Queries should be terse, keyword-heavy like real user searches: "k8s pod memory limits configuration", "quarterly revenue breakdown Q3"
- Connecting queries come from BM25 (keyword matching)—shared terms don't guarantee meaningful relationships. Scrutinize whether matched terms have the same meaning in both chunks.
- Same high-level entity ≠ valid connection. Two chunks mentioning "Q3 revenue" or "the API" need actual dependency, not just shared subject matter.
- **When in doubt, choose "No Valid Relationship."** Weak relationships produce bad questions.
- **Hard requirements for every question:**
  1. Question vocabulary matches source chunk better than target
  2. Source chunk contains explicit or inferrable path to target
  3. Answer cannot be complete without target chunk's information
  4. Target chunk's distinctive terms must be paraphrased (use hypernyms, describe function/outcome, genericize proper nouns)
- Rank and place your best question first if multiple q&a pairs are generated

## Step 1: Identify relationship type

Classify into exactly one category:

### Explicit Reference
One chunk mentions, names, or links to content that the other chunk *is*.

Signals: Direct references ("see X", "refer to X"), matching document/section names, links, citations, phrases like "as explained in..."

Example: Source says "Q3 planning doc references the customer research findings" → Target is the customer research report

If found: Note direction (A→B means A is source, B is target)

### Abstraction Levels
Same core information at different granularity. One summarizes/claims, the other details/proves.

Signals: Claim + evidence, code + rationale, concept + procedure, summary + full content

Example: Source says "Mediterranean diet has strong evidence for heart health" → Target has study details with participant data and outcomes

Ensure both chunks refer to the same topic/sub-topic, not something tangentially related. **Bidirectional questions possible**—either chunk can serve as source.

### No Valid Relationship

Choose this if:
- Chunks are unrelated or connection is superficial
- Connection requires excessive inference (more than one logical step / only tangentially related)
- Near-duplicates (multi-hop adds no value)
- Content is independently complete on similar subjects

## Step 2: Generate Questions

Skip if relationship type is "No Valid Relationship."

### Question Strategies

**Explicit Reference:** Frame question around the *context* in source where reference appears. Ask for what target provides using paraphrased terms.

Source: "Trip itinerary mentions the restaurant recommendations doc"
Target: [Restaurant list: Tsuta for ramen, Sukiyabashi for sushi...]
✓ "good food japan trip plan" — matches source context, needs target for specifics, no target keywords
✗ "Tsuta ramen Sukiyabashi sushi" — retrieves target directly, bypasses source
✗ "japan trip plan" — matches source, but has no relevance to the target

Source: "Customer onboarding improvements are detailed in the Q3 ops review"
Target: [Ops review data: automated welcome emails, support ticket reduction 60%, average onboarding time 3 days → 4 hours...]
✓ "what changed in customer onboarding process" — matches source context, needs target for specifics, no target keywords
✗ "welcome email automation support ticket reduction" — retrieves target directly, bypasses source
✗ "when was Q3 ops review published" — retrieves source, but asks about metadata, not onboarding details

**Abstraction Levels:** Use vocabulary from source chunk, require precision only target provides. Generate questions in both directions.

General: "The org restructure significantly improved cross-team collaboration"
Specific: [Survey data: cross-team project completion up 40%, meeting conflicts down 25%...]
✓ General→Specific: "measurable impact of reorg on team collaboration"
✓ Specific→General: "leadership claims about collaboration survey results"
✗ "cross-team project completion meeting conflicts" — retrieves specific directly, bypasses general
✗ "org restructure announcement" — retrieves general, but no indication that survey data is needed

General: "Our new caching layer reduced API latency dramatically"
Specific: [Redis implementation: cache hit rates 94%, p99 latency down from 800ms to 120ms, eviction policies...]
✓ General→Specific: "performance metrics for the API speed improvements"
✓ Specific→General: "engineering summary of Redis cache results"
✗ "Redis cache hit rate p99 latency" — retrieves specific directly, bypasses general
✗ "new caching layer" — retrieves general, but no indication that implementation details are needed

Return JSON with:
- thoughts: Analysis of the relationship. State the relationship type found (or why none exists). If relationship exists, identify source vs target and the linking mechanism.
- relationship_type: "explicit_reference" | "abstraction_levels" | "none"
- direction: "A_to_B" | "B_to_A" | "bidirectional" | null
- linking_info: Object describing the connection, or null if none. Structure depends on relationship_type:
    - For explicit_reference: {{ reference_text, source ("A"|"B"), target ("A"|"B") }}
    - For abstraction_levels: {{ general_chunk ("A"|"B"), specific_chunk ("A"|"B"), abstraction_link }}
- qa_pairs: List of QA pairs, or null if no valid multi-hop questions can be formed. Each pair contains:
    - question: Terse, keyword-focused multi-hop question
    - answer: Answer requiring synthesis of both chunks
"""

multi_hop_system_prompt = render_template(multi_hop_system_template, corpus_context)

multi_hop_user_template = """Analyze the connection between these chunks.

Connecting Queries: {connecting_queries}

<chunk_a_context_before>
{chunk_a_context_before}
</chunk_a_context_before>

<chunk_a>
{chunk_a}
</chunk_a>

<chunk_a_context_after>
{chunk_a_context_after}
</chunk_a_context_after>

---

<chunk_b_context_before>
{chunk_b_context_before}
</chunk_b_context_before>

<chunk_b>
{chunk_b}
</chunk_b>

<chunk_b_context_after>
{chunk_b_context_after}
</chunk_b_context_after>

Analyze whether there is a meaningful relationship between the chunks and whether multi-hop questions can be formed.

Return JSON with keys: thoughts, relationship_type, direction, linking_info, qa_pairs
"""


def generate_related_queries(
    chunk: Chunk,
    client,
    model: str,
    context_preview_chars: int = 200,
) -> tuple[str, str, str]:
    """Generate queries to find related chunks. Returns (queries, parsed_result)."""
    ctx = source.get_chunk_with_context(chunk, max_chars=context_preview_chars)
    user_prompt = render_template(related_chunk_user_template, ctx)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": related_chunk_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    response_text = completion.choices[0].message.content or ""
    return related_chunk_system_prompt, user_prompt, response_text


def generate_multi_hop_qa(
    chunk_a: Chunk,
    chunk_b: Chunk,
    connecting_queries: list[str],
    client,
    model: str,
    context_preview_chars: int = 200,
) -> tuple[str, str, str]:
    """Validate connection and generate QA pairs. Returns (system_prompt, user_prompt, response_text)."""
    # Get context for both chunks
    ctx_a = source.get_chunk_with_context(chunk_a, max_chars=context_preview_chars)
    ctx_b = source.get_chunk_with_context(chunk_b, max_chars=context_preview_chars)

    user_prompt = render_template(
        multi_hop_user_template,
        {
            "connecting_queries": connecting_queries,
            "chunk_a": ctx_a["chunk_content"],
            "chunk_a_context_before": ctx_a["prev_chunk_preview"],
            "chunk_a_context_after": ctx_a["next_chunk_preview"],
            "chunk_b": ctx_b["chunk_content"],
            "chunk_b_context_before": ctx_b["prev_chunk_preview"],
            "chunk_b_context_after": ctx_b["next_chunk_preview"],
        },
    )

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": multi_hop_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    response_text = completion.choices[0].message.content or ""
    return multi_hop_system_prompt, user_prompt, response_text


# Sample 2 chunks to test related chunk finding and multi-hop QA generation
sample_chunks = source.sample_chunks(2, min_chars=400)

for i, chunk_a in enumerate(sample_chunks, 1):
    print_sections((f"Chunk {i}, {len(chunk_a)} chars", chunk_a.chunk_str()))
    print(
        f"{'=' * 120}\nFinding related queries for SAMPLE {i} of {len(sample_chunks)}. This should take ~10s+..."
    )

    system_prompt_step_1, user_prompt_step_1, response = generate_related_queries(
        chunk_a, client, MODEL, CONTEXT_PREVIEW_CHARS_RELATED
    )

    confidence, queries = parse_related_queries_response(response)

    # # UNCOMMENT to print step 1 details
    # print_sections(
    #     # (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 System Prompt", system_prompt_step_1),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 User Prompt", user_prompt_step_1),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Step 1 Response", response),
    #     (f"SAMPLE {i} of {len(sample_chunks)} - Parsed Related Queries",
    #         f"Confidence: {confidence}\nQueries: {queries}",
    #     ),
    # )

    if confidence.lower() != "high":
        print("Skipping multi-hop generation due to low confidence in related queries.\n")
        continue

    # Search for related chunks
    search_results = source.search_related(chunk_a, queries, top_k=TOP_K_BM25_RESULTS)

    # Take the top related chunk
    for j, chunk_info in enumerate(search_results[:TOP_RELATED_CHUNKS], 1):
        chunk_b = chunk_info["chunk"]
        connecting_queries = chunk_info["queries"]

        print(
            f"{'-' * 120}\nGenerating multi-hop Q&A pair for SAMPLE {i}.{j} of {len(sample_chunks)}. This might take ~20s+..."
        )

        (
            system_prompt_step_2,
            user_prompt_step_2,
            response,
        ) = generate_multi_hop_qa(chunk_a, chunk_b, connecting_queries, client, MODEL)

        qa_pairs = parse_multi_hop_validation_response(response)

        if not qa_pairs:
            print("No QA pairs generated.\n")
            continue

        print_sections(
            # (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 System Prompt", system_prompt_step_2),
            (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 User Prompt", user_prompt_step_2),
            (f"SAMPLE {i}.{j} of {len(sample_chunks)} - Step 2 Response", response),
            # (
            #     f"SAMPLE {i}.{j} of {len(sample_chunks)} - Parsed Multi-hop QA",
            #     f"Generated {len(qa_pairs)} QA pairs: {qa_pairs}",
            # ),
        )

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                 Chunk 1, 1122 chars                                                  │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
{
  "file": "integrate/_snippets/install-unity.mdx",
  "h3": "Configuration options",
  "index": 1
}
### Configuration options  
All options below can be set via the Unity Inspector or in code:  
```csharp
PostHog.Setup(new PostHogConfig
{
// Required
ApiKey = "<ph_project_token>",

// Optional
Host = "<ph_client_api_host>",         // PostHog instance URL (default: https://us.i.posthog.com)
FlushAt = 20,                           // Events before auto-flush (default: 20)
FlushIntervalSeconds = 30,              // Seconds between flushes (default: 30)
MaxQueueSize = 1000,                    // Max queued events (default: 1000)
Ma

## 4. Batch Run

This section will process all chunks in batch to generate the full dataset.

**Configuration:**

- Number of single-hop request -> each request might generate 0-2 questions.
- Number of multi-hop request -> each request might generate 0-10 questions.

In [15]:
from synthetic_data_prep.qa_generation.helpers import (
    generate_multi_hop_batch,
    generate_single_hop_batch,
)
from synthetic_data_prep.qa_generation.storage import (
    qa_dataset_to_jsonl_bytes,
    save_qa_dataset,
    save_qa_dataset_jsonl,
)

# ===== CONFIGURATION =====
NUM_SINGLE_HOP_SAMPLES = 40
NUM_MULTI_HOP_SAMPLES = 5

# Maximum number of questions to use per chunk
# Set to None for no limit (use all generated questions)
MAX_QUESTIONS_PER_CHUNK = 2  # Set to 2 to get more chunk diversity

# Batch processing configuration
MAX_CONCURRENT = 10  # Number of concurrent LLM calls
MAX_TOKENS = 1000  # Max tokens per response
TIMEOUT = 60.0  # Request timeout in seconds

# Output paths
OUTPUT_PATH_YAML = "outputs/qa_dataset.yaml"  # Human-readable format
OUTPUT_PATH_JSONL = "outputs/qa_dataset.jsonl"  # HuggingFace-compatible format
# =========================


# Generate single-hop QA pairs in batch
print(f"\n{'=' * 120}\nGenerating single-hop QA pairs...\n{'=' * 120}")

single_hop_dataset = generate_single_hop_batch(
    source=source,
    client=client,
    model=MODEL,
    system_prompt=single_hop_system_prompt,
    user_template=single_hop_user_template,
    num_samples=NUM_SINGLE_HOP_SAMPLES,
    response_parser=parse_single_hop_response,
    context_preview_chars=CONTEXT_PREVIEW_CHARS,
    max_concurrent=MAX_CONCURRENT,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
    max_questions=MAX_QUESTIONS_PER_CHUNK,
)

print(f"\nSinfle hop dataset: {len(single_hop_dataset)} total data points\n")

# Generate multi-hop QA pairs in batch
print(f"\n{'=' * 120}\nGenerating multi-hop QA pairs...\n{'=' * 120}")

multi_hop_dataset = generate_multi_hop_batch(
    source=source,
    client=client,
    model=MODEL,
    related_query_system_prompt=related_chunk_system_prompt,
    related_query_user_template=related_chunk_user_template,
    multi_hop_system_prompt=multi_hop_system_prompt,
    multi_hop_user_template=multi_hop_user_template,
    num_samples=NUM_MULTI_HOP_SAMPLES,
    related_query_parser=parse_related_queries_response,
    multi_hop_parser=parse_multi_hop_validation_response,
    context_preview_chars=CONTEXT_PREVIEW_CHARS_RELATED,
    top_k_bm25=TOP_K_BM25_RESULTS,
    top_related_chunks=TOP_RELATED_CHUNKS,
    max_concurrent=MAX_CONCURRENT,
    max_tokens=MAX_TOKENS,
    timeout=TIMEOUT,
    max_questions=MAX_QUESTIONS_PER_CHUNK,
)

print(f"\nMulti hop dataset: {len(multi_hop_dataset)} total data points\n")

# Merge datasets and save
print(f"\n{'=' * 120}\nMerging and saving... {'=' * 120}")

combined_dataset = []
combined_dataset.extend(single_hop_dataset)
combined_dataset.extend(multi_hop_dataset)
print(f"\nCombined dataset: {len(combined_dataset)} total data points\n")

# Save in both formats
save_qa_dataset(combined_dataset, OUTPUT_PATH_YAML)
print(f"\nSaved YAML (human-readable) to {OUTPUT_PATH_YAML}")

save_qa_dataset_jsonl(combined_dataset, OUTPUT_PATH_JSONL)
print(f"Saved JSONL (HuggingFace-compatible) to {OUTPUT_PATH_JSONL}")

# # UNCOMMENT the code below to load the saved dataset
# loaded_dataset = load_qa_dataset(OUTPUT_PATH_YAML)
# print(f"\nLoaded dataset: {loaded_dataset.summary()}")


Generating single-hop QA pairs...


Processing prompts: 100%|██████████| 40/40 [01:08<00:00,  1.72s/it]



Sinfle hop dataset: 70 total data points


Generating multi-hop QA pairs...
Step 1: Generating related queries for 5 chunks...


Processing prompts: 100%|██████████| 5/5 [00:02<00:00,  1.74it/s]



Step 2: Validating 15 chunk pairs for multi-hop QA...


Processing prompts:   0%|          | 0/15 [00:00<?, ?it/s]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/home/angelpan/.pyenv/versions/3.12.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x7f8980142b80> is already entered
Processing prompts:   7%|▋         | 1/15 [00:12<02:57, 12.71s/it]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/home/angelpan/.pyenv/versions/3.12.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x7f8980142b80> is already entered
Processing prompts: 100%|██████████| 15/15 [00:23<00:00,  1.54s/it]
Task was destroyed but it is pending!
task: <Task pending name='Task-604' coro=<_async_in_context.


Multi hop dataset: 20 total data points


Merging and saving... ========================================================================================================================

Combined dataset: 90 total data points


Saved YAML (human-readable) to outputs/qa_dataset.yaml
Saved JSONL (HuggingFace-compatible) to outputs/qa_dataset.jsonl


#### SAGE Implementation

In [ ]:
question_generation_prompt = '''
Your task is to generate a complicated question that will require a search agent target_search_step search steps to answer by
gathering information using a search engine.
You will first reason about the initial document and plan for gathering comprehensive information inside <think> and </think>.
You will then make a tool call and it will return the top searched results to collect information.
You must conduct reasoning inside <think> and </think> first every time you get new information.
You will call the search engine for n_search_step steps. After n_search_step searches, you must provide the question inside
<question> and </question>, the answer inside <answer> and </answer>, and the answering step inside <answering steps>
and </answering steps>. You can use your own knowledge to construct the search query, but the final answer and each of the
answering step must be supported by the information you gathered from the search engine.
The question should be understandable standalone as the agent will use the question to search for information without access to
the initial document.
An example question: How much did the film in which Jake Gyllenhaal played his second lead role gross in its initial run at the
box office?
Make sure the answer is correct and **unique** for the question generated.
Initial document: context
'''

search_agent_prompt = '''
Answer the given question by using a search engine.
You will first reason about the question inside <think> and </think>, for instance, break down the question into multiple
sub-questions that you will search for.
You must call a search engine by <search> query </search> and it will return the top searched results between <information>
and </information>.
Try to formulate the search query in the form of a question.
After receiving the information, you must reason about it inside <think> and </think> before issuing a new query or providing
the final answer.
Each of your reasoning step should be grounded in the retrieved information. Do not use your own knowledge, but you can use
commonsense knowledge or arithmetic knowledge.
Do not use your own knowledge to write the query, the query should be based on the question and the retrieved documents.
Do not infer the entities in the question, but you can use the entities in the retrieved documents to write the query.
You can search as many times as your want. Try to break down the question for each search query and gather comprehensive
information.
If you have gathered enough information to answer the question, you can provide the answer to the query inside <answer> and
</answer>, without detailed illustrations.
Generate an answer based on the retrieved information, instead of your own knowledge.
This is an example answer: <answer>Beijing</answer>. Question: question
'''

llmaaj_prompt = '''
Judge whether the following [response] to [question] is correct or not based on the precise and unambiguous [correct_answer_list]
below. Each answer in the [correct_answer_list] is separated by a comma.
[question]: question
[response]: model_answer
Your judgment must be in the format and criteria specified below:
extracted_final_answer: The final exact answer extracted from the [response]. Put the extracted answer as ’None’ if there is no
exact, final answer to extract from the response.
[correct_answer_list]: gold_answer
reasoning: Explain why the extracted_final_answer is correct or incorrect based on [correct_answer_list], focusing only on if
there are meaningful differences between answer in the [correct_answer_list] and the extracted_final_answer. Focus on recall, i.e.
if the extracted_final_answer covers all the points in the answer in the [correct_answer_list]. It is ok if it provides more details.
It is also ok if the extracted_final_answer misses minor point from the correct_answer, as long as it is evident that they are
referring to the same thing. Do not comment on any background to the problem, do not attempt to solve the problem, do not
argue for any answer different than [correct_answer_list], focus only on whether the answers match. Ignore capitalization.
correct: Answer ’yes’ if extracted_final_answer matches any of the answers in [correct_answer_list] given above, or is within a
small margin of error for numerical problems. Answer ’no’ otherwise, i.e. if there is any inconsistency, ambiguity, non-equivalency,
or if the extracted answer is incorrect.
confidence: The extracted confidence score between 0% and 100% from [response]. Put 100 if there is no confidence score
available.
'''

execution_feedback_incorrect_prompt = '''
You will be given an output from a question generator agent, which generates a complicated question, answer pair; as well as the
output from a search agent, which attempts to solve the question generated in a fixed number of turns.
The answer from the search agent is not the same as the data generator agent. You task is to examine their traces and output the
correct question, answer pair based on their retrieved documents. You can update either the question, the answer or both.
You will first reason about why is there a discrepancy between the search agent’s answer and the data generator’s answer. Output
your reasoning trace inside <reason> and </reason>. You will then reason about how to update the question answer pair
to make sure it is correct and requires the agent target_step search step to answer. A search step is defined as a call to the
search tool. Output your reasoning trace inside <think> and </think>. For factual information, you should ONLY rely on the
context provided for the data generator agent and the documents retrieved by both the data generator and search agent (inside
<information> and </information>).
If you find it non-trivial to update just the question and answer, you can generate a new question answer pair ONLY based on the
retrieved documents.
The updated question should require the search agent at least target_step search steps to answer. However, the answer should
be short, such as an entity, a date or a number. The question should be understandable standalone, as the search agent will solve
the question without access to the documents (they will need to search for them).
When you are ready to provide the new question, answer pair, you can provide the question inside <question> and </question>,
the answer inside <answer> and </answer>, and the search step inside <search steps> and </search steps>. For each search
step, output the exact search question; the sub-answer to the search question; and the retrieved document from the search agent
and data generator agent’s output that supports the sub-answer. Make sure each step is absolutely needed to answer the question
and there is no short cut. Tip: use retrieved document from different steps so avoid two sub-queries being solved by one search
query.
# Data generator agent
Prompt: data_generator_agent_prompt
Agent’s output: data_generator_agent_response
# Search agent
Prompt: search_agent_prompt
Agent’s output: search_agent_response
# Your output
'''

execution_feedback_too_easy_prompt = '''
You will be given an output from a question generator agent, which generates a complicated question, answer pair to be solved by
a search agent for at least target_step **search** steps; as well as the output from a search agent, which attempts to solve the
question generated. The search agent is able to solve the question in less thantarget_step search steps. Your task is to update
the question so that it requires the search agent more steps to solve.
You will first reason about why the search agent is able to solve the question in fewer steps. Output your reasoning trace inside
<reason> and </reason>. You will then reason about how to update the question so that it will require more search steps. For
factual information, you should ONLY rely on the context provided for the data generator agent and the documents retrieved by
both the data generator and search agent (inside <information> and </information>), without relying on other information
not in the retrieved context. Output your reasoning trace inside <think> and </think>. If you find it non-trivial to update the
plan, you can generate a new question answer pair ONLY based on the retrieved documents.
The updated question should require the search agent at least target_step search steps to answer. Note that some of the
answering steps do not involve search and thus do not count. However, the answer should be short, such as an entity, a date or a
number. The question should be understandable standalone, as the agent will solve the question without access to the documents
(they will need to search for them).
When you are ready to provide the new question, answer pair, you can provide the question inside <question> and </question>,
the answer inside <answer> and </answer>, and the search step inside <search steps> and </search steps>. For each search
step, output the exact search question; the sub-answer to the search question; and the retrieved document from the search agent
and data generator agent’s output that supports the sub-answer. Make sure each step is absolutely needed to answer the question
and there is no short cut. Tip: use retrieved document from different steps so avoid two sub-queries being solved by one search
query.
# Data generator agent
Prompt: data_generator_agent_prompt
Agent’s output: data_generator_agent_response
# Search agent
Prompt: search_agent_prompt
Agent’s output: search_agent_response
# Your output
'''

reasoning_analysis_prompt = '''
You will be given a question and a trace of a search agent solving this question. You will analyze and categorize the behavior for
each of the thinking step.
Below are some example categories, please feel free to propose new ones as you see appropriate:
- Information inference: the agent makes an inference based on the piece of information in the retrieved document in its reasoning.
- Conflict resolution: the agent reasons about conflicting information in the documents and makes a decision.
- Calculation: The agent performs numerical calculation.
- Temporal reasoning: The agent performs temporal reasoning, such as deriving duration between two dates.
- Self-correction: The agent recognizes that the previous search failed to yield the required information (a list of stations). It
re-evaluates its state, confirms its goal, and decides to try a more specific search query.
- Hypothesis Generation: The agent makes a guess / hypothesis that is not grounded in the retrieved documents.
Output format: only return the list of strategies for each step:
- Step i: [list of strategies]
- Step i+1: [list of strategies]
Question: question Agent’s reasoning trace: agent_trace
'''

In [125]:
from synthetic_data_prep.envs.cgft_search_env import CgftSearchEnv
BASE_URL = "https://app.cgft.io"
AZURE_API_KEY = 'EldWZtATVrvUQMKTjgAQOHeadaoiHJoD8SrorScBm6IUn44yARevJQQJ99CBACHYHv6XJ3w3AAAAACOGUUWL'
AZURE_URL = 'https://expt-platform-foundry.openai.azure.com/openai/v1'
CORPUS_ID = source.corpus_id
QGEN_MODEL = "gpt-5.2"
class QuestionGenEnv(CgftSearchEnv):
    system_prompt = question_generation_prompt 

qgenv = QuestionGenEnv(api_key=AZURE_API_KEY, corpus_id=CORPUS_ID, base_url=AZURE_URL)
print(await qgenv.get_system_prompt(True))


Your task is to generate a complicated question that will require a search agent target_search_step search steps to answer by
gathering information using a search engine.
You will first reason about the initial document and plan for gathering comprehensive information inside <think> and </think>.
You will then make a tool call and it will return the top searched results to collect information.
You must conduct reasoning inside <think> and </think> first every time you get new information.
You will call the search engine for n_search_step steps. After n_search_step searches, you must provide the question inside
<question> and </question>, the answer inside <answer> and </answer>, and the answering step inside <answering steps>
and </answering steps>. You can use your own knowledge to construct the search query, but the final answer and each of the
answering step must be supported by the information you gathered from the search engine.
The question should be understandable standalone as

In [127]:
source.corpus_id

'2e63166e-8ace-42e4-b2c9-1f9a92e96c99'

In [126]:
import tempfile
from pathlib import Path

import synthetic_data_prep
from benchmax.bundle.bundler import bundle_env, write_bundle_files
from synthetic_data_prep.trainer.client import RolloutClient

# 1) Bundle the env locally (no upload needed)
constructor_args = {
    "api_key": API_KEY,
    "corpus_id": CORPUS_ID,
    "base_url": BASE_URL,
}

bundle = bundle_env(
    QuestionGenEnv,
    pip_dependencies=["aiohttp"],
    local_modules=[synthetic_data_prep],
    constructor_args=constructor_args,
)

with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    pkl_path = tmp_path / "env-cls.pkl"
    meta_path = tmp_path / "env-metadata.json"
    write_bundle_files(bundle, pkl_path, meta_path)
    env_cls_bytes = pkl_path.read_bytes()
    env_metadata_bytes = meta_path.read_bytes()

# 2) Build a sample example
sample_chunks = source.sample_chunks(2, min_chars=400)

raw_example = {
    "question": f"Initial document: {sample_chunks[0]}\nn_search_step: 2",
    "answer": ""
}

# 3) Run one agent rollout (bytes sent inline — no blob upload)
rollout = RolloutClient(api_key=API_KEY)
final = rollout.stream_rollout(
    raw_example=raw_example,
    env_cls_bytes=env_cls_bytes,
    env_metadata_bytes=env_metadata_bytes,
    max_turns=8,
    max_tool_calls=12,
    capture_messages=True,
    llm_api_key=AZURE_API_KEY,
    llm_base_url=AZURE_URL,
    llm_model=QGEN_MODEL,
    max_completion_tokens=5128 
)

print(final.get("success"))
print(final.get("rewards"))
print(final.get("final_assistant_text"))

  [ex 0] → rollout_started
  [ex 0] → message [user]: Initial document: { "file": "posthog-ai/faq.mdx", "h2": "Have a question? Ask PostHog AI", "index": 0 } --- title:…
  [ex 0] → message [assistant]: <think>Need 2 searches. Ask about specific providers and access controls etc. Create compound Q requiring confirm from…
  [ex 0] → message [user]: Tool: search 1. — (score: 40.16) Content: --- title: PostHog AI FAQ --- ## Have a question? Ask PostHog AI <AskAIInput…
  [ex 0] → message [assistant]: <question> For PostHog AI (as described in PostHog’s AI FAQ/overview docs), list: (a) every category of project data…
  [ex 0] → reward: {'chunk_overlap_reward_function': 0.0}
  [ex 0] → rollout_completed  ✔  success  rewards={'chunk_overlap_reward_function': 0.0}  error=None
True
{'chunk_overlap_reward_function': 0.0}
<question>
For PostHog AI (as described in PostHog’s AI FAQ/overview docs), list: (a) every category of project data PostHog AI can read “when doing research,” (b) what exactly r

<!-- ### 4.1 Filter Agent

Goal: discard questions that the target model can answer from pretraining knowledge alone, without any document access.

The Filter Agent takes a question, runs it against your target LLM with no context, then uses a Judge LLM to compare that answer against the reference answer. If they're semantically equivalent the question is too easy and gets filtered out. -->

In [16]:
from synthetic_data_prep.qa_generation.models import QADataPoint

def run_filter_agent(
    qa_pair: QADataPoint,
    target_model: str = "gpt-5-nano",  # cheaper model for the student
    judge_model: str = "gpt-5-mini",        # stronger model for judging
) -> dict:

    # Step 1: attempt to answer without any document access
    no_context_response = client.chat.completions.create(
        model=target_model,
        messages=[
            {
                "role": "system",
                "content": "Answer the following question as best you can. "
                           "Do not say you don't know — make your best attempt."
            },
            {
                "role": "user",
                "content": qa_pair["question"]
            }
        ]
    )
    attempted_answer = no_context_response.choices[0].message.content

    # Step 2: judge whether the attempted answer matches reference
    judge_response = client.chat.completions.create(
        model=judge_model,
        messages=[
            {
                "role": "system",
                "content": """You are evaluating whether two answers convey 
                the same information. 
                Respond with JSON only:
                {
                  "is_equivalent": true/false,
                  "reasoning": "brief explanation"
                }
                Be strict — partial matches or vague answers should 
                be marked false."""
            },
            {
                "role": "user",
                "content": f"""Reference answer: {qa_pair['answer']}
                
Attempted answer: {attempted_answer}

Are these equivalent?"""
            }
        ],
        response_format={"type": "json_object"}
    )

    import json
    judgment = json.loads(judge_response.choices[0].message.content)

    return {
        **qa_pair,
        "filter_status": "rejected" if judgment["is_equivalent"] else "passed",
        "filter_reasoning": judgment["reasoning"],
        "no_context_answer": attempted_answer,
    }


def run_filter_pipeline(qa_pairs: list[QADataPoint]) -> tuple[list[QADataPoint], dict]:
    results = [run_filter_agent(pair) for pair in qa_pairs]
    filtered_pairs = [pair for i, pair in enumerate(qa_pairs) if results[i] == "passed"]
    rejected = [r for r in results if r["filter_status"] == "rejected"]

    stats = {
        "total": len(results),
        "passed": len(filtered_pairs),
        "rejected": len(rejected),
        "pass_rate": len(filtered_pairs) / len(results) if results else 0,
        "rejection_reasons": [r["filter_reasoning"] for r in rejected]
    }

    return filtered_pairs, stats

In [17]:
filtered_dataset, filtering_stats = run_filter_pipeline(combined_dataset)

### 4.1 Evaluate Q&A Quality with Ragas

[Ragas](https://docs.ragas.io) provides LLM-based metrics to assess the quality of generated Q&A pairs:

- **Faithfulness** -- Is the answer grounded in the reference chunks? Catches hallucinated content.
- **Context Precision** -- Are the reference chunks relevant to the answer? Detects poorly matched chunks.

#### 4.1.1 Save data as Ragas Dataset

In [77]:
import os
os.environ["RAGAS_DO_NOT_TRACK"] = "true"

In [ ]:
# Save data to ragas dataset
from ragas import Dataset
def save_qa_dataset_to_ragas_dataset(qa_dataset: list[QADataPoint]) -> Dataset:
    dataset = Dataset(
        name="qa_pairs_tier1",
        backend="local/csv",
        root_dir="./eval",
    )
    for pair in qa_dataset:
        dataset.append({
            "question": pair["question"],
            "reference_answer": pair["answer"],
            # Store chunks as joined string — CSV doesn't support lists
            # we'll split on load
            "source_chunks": "|||".join(
                c["content"] for c in pair["reference_chunks"]
            ),
            "chunk_ids": "|||".join(
                c["id"] for c in pair["reference_chunks"]
            ),
            "qa_type": pair["qa_type"],
            "min_hop_count": pair["min_hop_count"],
            "is_co_located": pair["is_co_located"],
        })
    dataset.save()
    return dataset

In [79]:
ragas_dataset = save_qa_dataset_to_ragas_dataset(filtered_dataset)

#### 4.1.2 Define Metrics for Intrinsic Evaluation

In [ ]:
from openai import AsyncOpenAI, AsyncAzureOpenAI
from ragas.embeddings import OpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextRelevance,
    DomainSpecificRubrics
)

# ragas metrics use async internally, so we derive an AsyncOpenAI client
# from the existing sync client's credentials
AZURE_API_KEY = 'EldWZtATVrvUQMKTjgAQOHeadaoiHJoD8SrorScBm6IUn44yARevJQQJ99CBACHYHv6XJ3w3AAAAACOGUUWL'
AZURE_URL = 'https://expt-platform-foundry.openai.azure.com/'
ragas_client = AsyncOpenAI(api_key=client.api_key, base_url=str(client.base_url))
embeddings_client = AsyncAzureOpenAI(api_key=AZURE_API_KEY, azure_endpoint=AZURE_URL, api_version='2025-12-11')


RAGAS_JUDGE_MODEL = "gpt-5.2"

# ===== MODIFY HERE =====
RAGAS_MAX_TOKENS = 4096  # default is 1024; increase if faithfulness fails on long chunks
RAGAS_BATCH_SIZE = 5     # concurrent requests per batch
# ========================

ragas_llm = llm_factory(MODEL, client=ragas_client, max_tokens=RAGAS_MAX_TOKENS)
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", client=embeddings_client)

faithfulness_metric = Faithfulness(llm=ragas_llm)
relevancy_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
contextual_metric = ContextRelevance(llm=ragas_llm)

difficulty_metric = DomainSpecificRubrics(
    name="question_difficulty",
    llm=ragas_llm,
    rubrics={
        "score1_description": "Answerable from a single chunk with direct lookup, no reasoning required.",
        "score2_description": "Requires connecting information across 2-3 chunks or basic inference.",
        "score3_description": "Requires multi-step reasoning across 4+ chunks or synthesis of conflicting information.",
    }
)

agnosticism_metric = DomainSpecificRubrics(
    name="context_independence",
    llm=ragas_llm,
    rubrics={
        "score1_description": "Question directly references the source document (e.g. 'in this document', 'as described above') — unusable standalone.",
        "score2_description": "Question implies a specific document context but could plausibly stand alone.",
        "score3_description": "Fully self-contained question with no reference to source material.",
    }
)

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def compute_diversity(questions: list[str]) -> float:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(questions)
    sim_matrix = cosine_similarity(embeddings)
    n = len(questions)
    mask = ~np.eye(n, dtype=bool)
    return float(1 - sim_matrix[mask].mean())


DEBUG:instructor:Patching `client.chat.completions.create` with mode=<Mode.JSON: 'json_mode'>
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): t.explodinggradients.com:443
DEBUG:ragas._analytics:Tracking Error: HTTPSConnectionPool(host='t.explodinggradients.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)')))


#### 4.1.3 Define Experiements

In [102]:
from ragas import experiment, Dataset

@experiment()
async def tier1_baseline(row: dict) -> dict:
    """
    Tier 1 baseline experiment.
    Scores each QA pair on all intrinsic metrics.
    """
    chunks = row["source_chunks"].split("|||")

    # Score all metrics
    faithfulness = await faithfulness_metric.ascore(
        user_input=row["question"],
        response=row["reference_answer"],
        retrieved_contexts=chunks,
    )
    # relevancy = await relevancy_metric.ascore(
    #     user_input=row["question"],
    #     response=row["reference_answer"]
    # )
    context = await contextual_metric.ascore(
        user_input=row["question"],
        retrieved_contexts=chunks,
    )
    difficulty = await difficulty_metric.ascore(
        user_input=row["question"],
        response=row["reference_answer"],
        retrieved_contexts=chunks,
    )
    agnosticism = await agnosticism_metric.ascore(
        user_input=row["question"],
        retrieved_contexts=chunks,
    )

    return {
        **row,
        "faithfulness": faithfulness.value,
        # "relevancy": relevancy.value,
        "context_relevance": context.value,
        "difficulty": difficulty.value,
        "context_independence": agnosticism.value,
        "experiment_name": "tier1_baseline",
    }

In [99]:
@experiment()
async def tier1_improved_prompt_v2(row: dict) -> dict:
    # Same scoring logic, different name
    # The name is what differentiates runs in your experiments/ folder
    ...
    return {
        **row,
        # all scores same as above
        "experiment_name": "tier1_improved_prompt_v2",
    }

#### 4.1.3 Run Ragas Experiment

In [119]:
import logging
import traceback
from transformers import logging as hf_logging


# Enable debug logging to see all HTTP traffic
logging.basicConfig(level=logging.WARN)

# Also enable httpx logging specifically (OpenAI SDK uses httpx under the hood)
logging.getLogger("httpx").setLevel(logging.WARN)
logging.getLogger("httpcore").setLevel(logging.WARN)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
hf_logging.set_verbosity_error()

In [ ]:
import asyncio
import pandas as pd

async def run_eval(experiment_fn, dataset_name: str = "qa_pairs_tier1"):
    dataset = Dataset.load(
        name=dataset_name,
        backend="local/csv",
        root_dir="./eval",
    )
    results = await experiment_fn.arun(dataset)
    return results


# Run baseline
asyncio.run(run_eval(tier1_baseline))

# # After tuning, run new version
# asyncio.run(run_eval(tier1_improved_prompt_v2))

Running experiment:   0%|          | 0/14 [00:00<?, ?it/s]DEBUG:instructor:Instructor Request: mode.value='json_mode', response_model=<class 'ragas.metrics.collections.faithfulness.util.StatementGeneratorOutput'>, new_kwargs={'messages': [{'role': 'system', 'content': '\n        As a genius expert, your task is to understand the content and provide\n        the parsed objects in json that match the following json_schema:\n\n\n        {\n  "description": "Structured output for statement generation.",\n  "properties": {\n    "statements": {\n      "description": "The generated statements from the answer",\n      "items": {\n        "type": "string"\n      },\n      "title": "Statements",\n      "type": "array"\n    }\n  },\n  "required": [\n    "statements"\n  ],\n  "title": "StatementGeneratorOutput",\n  "type": "object"\n}\n\n        Make sure to return an instance of the JSON, not the schema itself\n'}, {'role': 'user', 'content': 'Given a question and an answer, analyze the complexit

Experiment(name=lucid_hoare,  len=14)

In [121]:
# Compare — RAGAS auto-saves CSVs to eval/experiments/
# Load both and diff
def show_metrics(file_a: str, qa_dataset: QADataset):
    df_a = pd.read_csv(f"eval/experiments/{file_a}")

    metrics = [
        "diversity", "faithfulness", "relevancy", "factual_correctness",
        "context_relevance", "difficulty", "context_independence"
    ]

    print(f"{'Metric':<25} {'Baseline':>10}")
    print("-" * 55)
    for m in metrics:
        if m == "diversity":
            diversity = compute_diversity([p.question for p in qa_dataset])
            print(f"{m:<25} {diversity:>10.3f}")

        elif m in df_a.columns:
            a_mean = df_a[m].mean()
            print(f"{m:<25} {a_mean:>10.3f}")


In [122]:
show_metrics("lucid_hoare.csv", filtered_dataset)

Metric                      Baseline
-------------------------------------------------------


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 701.95it/s, Materializing param=pooler.dense.weight]                             


diversity                      0.631
faithfulness                   0.952
context_relevance              0.964
difficulty                     1.857
context_independence           2.714


## 5. Upload Q&A Dataset

Upload the generated Q&A pairs to storage for model training and evaluation.

The dataset is uploaded in JSONL format (HuggingFace-compatible) to the trainer mounted storage.


In [ ]:
from synthetic_data_prep.trainer import StorageClient

# ===== MODIFY HERE: Configure storage settings =====
# Note: You can use a different base URL if storage endpoint differs
STORAGE_BASE_URL = CORPUS_BASE_URL
# ===== END OF MODIFY SECTION =====

storage_client = StorageClient(api_key=API_KEY, base_url=STORAGE_BASE_URL)

# Upload the generated dataset as JSONL
print(f"Uploading {len(combined_dataset)} QA pairs for corpus {corpus.id}...")

dataset_path = f"datasets/search/{corpus.id}/qa-dataset.jsonl"

try:
    result = storage_client.upload_file(
        path=dataset_path,
        content=qa_dataset_to_jsonl_bytes(combined_dataset),
        mime_type="application/jsonl",
    )
    print("\nUpload successful!")
    print(f"  Blob path: {result['blobPath']}")
except Exception as e:
    print(f"\nUpload failed: {e}")

# # UNCOMMENT to upload from a saved file instead
# try:
#     result = storage_client.upload_local_file(
#         path=dataset_path,
#         file_path=OUTPUT_PATH_JSONL,
#     )
#     print("\nUpload successful!")
#     print(f"  Blob path: {result['blobPath']}")
# except Exception as e:
#     print(f"\nUpload failed: {e}")

## 5. RL Environment

This is the search environment that the trained LLM will be interacting with via tool calls. In an environment, we define the system prompt, tools, and reward functions. The reward function is executed at the end of the rollout (when the model reports the answer) and the score will be used to optimize the model

For the search environment, our tool is performing BM25 search on the corpus. (TODO: Need to implement llm judge rubric + reference chunk overlap calc) Our reward function ...


In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs import CorporaSearchEnv

Now that we have defined our search environment, let's bundle it and upload to the trainer storage.


In [ ]:
from synthetic_data_prep.trainer import StorageClient

# ===== MODIFY HERE: Configure storage settings =====
# Note: You can use a different base URL if storage endpoint differs
API_KEY = "sk_tXQpbKJEFGrWkkbHqBNPSNfSEfridmclDvTXAGXslJSmVOYFJlWDrfLLbuDakZTx"
STORAGE_BASE_URL = "https://app.cgft.io"
# ===== END OF MODIFY SECTION =====

storage_client = StorageClient(api_key=API_KEY, base_url=STORAGE_BASE_URL)

dataset_path = f"datasets/search/09bfcf73-ed4c-4e23-993e-42481bb177e4/qa-dataset.jsonl"

In [ ]:
import hashlib
import json

from benchmax.bundle.bundler import bundle_env
from benchmax.bundle.validator import validate_payload

constructor_args = {
    "api_key": API_KEY,
    "corpus_id": source.corpus_id,
    "base_url": "https://app.cgft.io",
    "dataset_path": f"~/user-data/{dataset_path}",
    # include ~/user-data/ prefix to index since that is where your storage would be mounted
}

payload = bundle_env(
    CorporaSearchEnv,
    pip_dependencies=["aiohttp"],  # Dependencies required for search functionality
    local_modules=[synthetic_data_prep],
    extra_metadata={"description": "Search env with local helpers"},
)

print(f"Payload size: {len(payload.to_bytes()) / 1024:.2f} KB")
print(f"Python version: {payload.python_version}")
print(f"Dependencies: {payload.pip_dependencies}")


# # TODO: Make this run the full eval loop to test the behavior e2e
# print(
#     "\nRunning validation in fresh environment to ensure all dependencies are correctly declared."
#     "\nThis will take ~1 min..."
# )
# warnings = validate_payload(
#     payload,
#     constructor_args=constructor_args,
# )
# if warnings:
#     print(f"Warnings: {warnings}")
# else:
#     print("Isolated validation passed!")

# Upload env
payload_bytes = payload.to_bytes()

# Calculate hash from content
content_hash = hashlib.sha256(payload_bytes).hexdigest()[:8]

# Upload to storage with hash in filename
env_path = f"envs/search/{content_hash}/search-env-cls.bmxp"
env_result = storage_client.upload_file(
    path=env_path,
    content=payload_bytes,
    mime_type="application/octet-stream",
)

env_args_path = f"envs/search/{content_hash}/search-env-kwargs.json"
env_args_bytes = json.dumps(constructor_args, indent=2).encode("utf-8")
env_args_result = storage_client.upload_file(
    path=env_args_path,
    content=env_args_bytes,
    mime_type="application/json",
)

print(f"Env uploaded successfully to {env_result['blobPath']}")
print(f"Env args uploaded successfully to {env_args_result['blobPath']}")

## 6. Start Training Job

Launch a training job using the uploaded dataset.


In [ ]:
from synthetic_data_prep.trainer import TrainerClient

# ===== MODIFY HERE: Configure trainer settings =====
TRAINER_API_KEY = API_KEY
TRAINER_BASE_URL = "https://app.cgft.io"
# ===== END OF MODIFY SECTION =====

trainer_client = TrainerClient(api_key=TRAINER_API_KEY, base_url=TRAINER_BASE_URL)

# Launch training experiment with the uploaded environment
try:
    experiment_id = trainer_client.launch_experiment(
        experiment_type="search",
        env_cls_path=f"~/user-data/{env_result['blobPath']}",
        env_kwargs_path=f"~/user-data/{env_args_result['blobPath']}",
    )
    print("Training experiment launched!")
    print(f"  Experiment ID: {experiment_id}")
except Exception as e:
    print(f"\nFailed to launch training experiment: {e}")